# CLIP Selective INT8/FP16 Optimization with TensorRT

This notebook provides a Google Colab entry point for reproducing the CLIP ViT-B/16 mixed-precision optimization pipeline.

The project applies selective INT8 quantization, exports a mixed INT8/FP16 ONNX model, builds TensorRT inference engines, and evaluates road-scene classification accuracy and GPU inference performance.

The implementation is maintained in `src/pipeline.py`; this notebook only configures and executes it.

**Reference environment:** NVIDIA Tesla T4 GPU, TensorRT 11.4.

Results may vary across GPUs, software versions, and runtime conditions.

In [ ]:
import sys
import subprocess
import os
from pathlib import Path

REPO = Path("/content/clip-tensorrt-optimization")

assert Path("/content").exists(), "This notebook requires Google Colab."

assert subprocess.run(
    ["nvidia-smi"],
    capture_output=True
).returncode == 0, "Enable a GPU runtime (T4 recommended)."

assert (REPO / "requirements.txt").exists(), (
    "Repository not found. Clone it first:\n"
    "!git clone https://github.com/suyashdhir/clip-tensorrt-optimization.git "
    "/content/clip-tensorrt-optimization"
)

# Store Hugging Face cache locally
os.environ["HF_HOME"] = "/content/hf"
os.environ["HF_HUB_CACHE"] = "/content/hf/hub"

print("Repository found:", REPO)
print("GPU runtime available.")
print("Hugging Face cache configured.")
print("\nInstall dependencies using:")
print(f"!{sys.executable} -m pip install -r {REPO / 'requirements.txt'}")

## Execute the Reproducible Pipeline

The pipeline runs six stages:

1. Prepare the calibration and evaluation datasets.
2. Evaluate the FP32 baseline.
3. Calibrate the selective INT8 quantization model.
4. Export and convert the model to mixed INT8/FP16 ONNX.
5. Build the TensorRT inference engines.
6. Evaluate accuracy and benchmark GPU inference.

Generated artifacts are saved persistently in Google Drive.

The pipeline supports resuming completed stages, so interrupted runs do not necessarily require starting over.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# feel free to change the repository name and path to your own Google Drive folder
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/clip_tensorrt_optimization"
)

print("Project workspace:", PROJECT_ROOT)

subprocess.run(
    [
        sys.executable,
        "-u",
        "-m",
        "src.pipeline",
        "--root",
        str(PROJECT_ROOT),
        "--stage",
        "all"
    ],
    cwd=str(REPO),
    check=True
)

print("\nPipeline completed successfully.")

## Saved Results

The pipeline saves evaluation metrics, predictions, SHA-256 artifact-integrity records, and GPU benchmark measurements.

Results are stored under:

`<PROJECT_ROOT>/results/`

Calibration checkpoints, ONNX models, and TensorRT engines are generated under the configured project workspace.

The repository's `results/` directory contains reference benchmark data and is separate from the newly generated results in Google Drive.

The authoritative pipeline implementation is located in `src/`.

In [ ]:
import json
import pandas as pd

RESULTS_DIR = PROJECT_ROOT / "results"

print("Results directory:", RESULTS_DIR)

models = {
    "FP16 TensorRT": "clip_baseline_fp16_metrics.json",
    "Mixed INT8/FP16 TensorRT": "clip_attention_fc1_int8_fp16_repaired_metrics.json"
}

for model_name, filename in models.items():
    path = RESULTS_DIR / filename

    print(f"\n=== {model_name} ===")

    if path.exists():
        metrics = json.loads(path.read_text())

        print(f"Accuracy: {metrics['accuracy']:.2%}")
        print(f"Macro F1: {metrics['macro_f1']:.4f}")
        print("Confusion matrix:")
        for row in metrics["confusion_matrix"]:
            print(row)
    else:
        print("Metrics file not found:", path)

comparison_path = RESULTS_DIR / "final_trt_comparison.csv"

if comparison_path.exists():
    print("\n=== TensorRT Benchmark Comparison ===")
    display(pd.read_csv(comparison_path))
else:
    print("\nComparison CSV not found.")